In [ ]:
import os
import tarfile

tar_path = "CUB_200_2011.tgz"
extract_dir = "./raw_cub"
dataset_url = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"

# 1. Download directly from Caltech server using wget
if not os.path.exists(tar_path):
    print("Downloading CUB-200 dataset... Please wait (~30 seconds).")
    !wget --no-check-certificate {dataset_url} -O {tar_path}
    print("Download finished!")
else:
    print("Dataset tar archive already exists!")

# 2. Extract archive
if not os.path.exists(extract_dir):
    print("Extracting files...")
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_dir)
    print("Extraction finished!")
else:
    print("Files already extracted!")

In [ ]:
import os
import shutil

source_images_path = os.path.join(extract_dir, "CUB_200_2011", "images")
dataset_dir = "./dataset_20_species"

if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
os.makedirs(dataset_dir)

# Get all species folders available in the dataset
all_available_species = sorted(os.listdir(source_images_path))

# Keywords relevant to BD birds/genera
bd_keywords = [
    'Crow', 'Kingfisher', 'Hummingbird', 'Mallard', 'Warbler',
    'Towhee', 'Jay', 'Creeper', 'Waxwing', 'Cuckoo',
    'Thrush', 'Woodpecker', 'Wren', 'Vireo', 'Catbird',
    'Meadowlark', 'Blackbird', 'Gull', 'Tern', 'Pelican'
]

# Match available species folders by keywords up to 20 species
selected_species = []
for species_folder in all_available_species:
    for kw in bd_keywords:
        if kw.lower() in species_folder.lower():
            if species_folder not in selected_species:
                selected_species.append(species_folder)
            break
    if len(selected_species) == 20:
        break

# Copy the matched 20 species
copied_count = 0
for species in selected_species:
    src = os.path.join(source_images_path, species)
    dst = os.path.join(dataset_dir, species)
    shutil.copytree(src, dst)
    copied_count += 1

print(f"Successfully matched and copied {copied_count} species into '{dataset_dir}'!\n")
print("Selected Species:")
for idx, sp in enumerate(selected_species, 1):
    print(f"{idx:2d}. {sp}")

In [ ]:
import os
from PIL import Image

valid_images = 0
corrupted_images = 0

for root, dirs, files in os.walk("./dataset_20_species"):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    img.verify()
                valid_images += 1
            except Exception as e:
                print(f"Removing corrupted image: {file_path}")
                os.remove(file_path)
                corrupted_images += 1

print("\n--- Image Integrity Check ---")
print(f"Healthy Valid Images: {valid_images}")
print(f"Corrupted Images Removed: {corrupted_images}")

In [ ]:
import os

dataset_dir = "./dataset_20_species"

print(f"{'Species Folder Name':<35} | {'Image Count'}")
print("-" * 52)

total = 0
for species in sorted(os.listdir(dataset_dir)):
    species_path = os.path.join(dataset_dir, species)
    if os.path.isdir(species_path):
        count = len(os.listdir(species_path))
        total += count
        print(f"{species:<35} | {count}")

print("-" * 52)
print(f"Total Species Folders: {len(os.listdir(dataset_dir))}")
print(f"Total Clean Images: {total}")

In [ ]:
!find / -name "*.ipynb" 2>/dev/null